# Lab 1 · Your first DuckDB query

**Due:** Sunday, October 25 at 11:59 PM

In this lab you will:
1. Connect to a sample dataset with DuckDB
2. Run your first SQL queries
3. Record three observations about the data

This is the **in-class** part of Lab 1. The at-home part (your own data, plus a lifecycle map) is described in `labs/lab1.md`.

**How to run a cell:** click inside it and press **Shift + Enter**. Run the cells in order, top to bottom.
The first time, VS Code may ask you to **Select Kernel** → choose **Python Environments** → the Python 3.12 option.

## 1. Connect to DuckDB

DuckDB is a database that runs *inside* your Python session: there's no server to start and no password.
The cell below opens a database file called `lab1.duckdb` in your repo (it's created the first time you run this).

In [1]:
from pathlib import Path
import duckdb

# Find the repo folder, whether this notebook runs from the repo root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "penguins.csv"

con = duckdb.connect(str(ROOT / "lab1.duckdb"))
print("Connected. Sample data at:", DATA.relative_to(ROOT))

Connected. Sample data at: data/penguins.csv


## 2. Look at the raw file

DuckDB can query a CSV file directly, before it's loaded into a table. `LIMIT 5` shows only the first five rows.

In [2]:
con.sql(f"SELECT * FROM '{DATA}' LIMIT 5")

┌─────────┬───────────┬────────────────┬───────────────┬───────────────────┬─────────────┬─────────┬───────┐
│ species │  island   │ bill_length_mm │ bill_depth_mm │ flipper_length_mm │ body_mass_g │   sex   │ year  │
│ varchar │  varchar  │    varchar     │    varchar    │      varchar      │   varchar   │ varchar │ int64 │
├─────────┼───────────┼────────────────┼───────────────┼───────────────────┼─────────────┼─────────┼───────┤
│ Adelie  │ Torgersen │ 39.1           │ 18.7          │ 181               │ 3750        │ male    │  2007 │
│ Adelie  │ Torgersen │ 39.5           │ 17.4          │ 186               │ 3800        │ female  │  2007 │
│ Adelie  │ Torgersen │ 40.3           │ 18            │ 195               │ 3250        │ female  │  2007 │
│ Adelie  │ Torgersen │ NA             │ NA            │ NA                │ NA          │ NA      │  2007 │
│ Adelie  │ Torgersen │ 36.7           │ 19.3          │ 193               │ 3450        │ female  │  2007 │
└─────────┴────────

## 3. Load it into a table

Now copy the file into a proper table called `penguins`. Once it's a table, it lives in `lab1.duckdb` and you can query it by name.
`CREATE OR REPLACE` means you can rerun this cell safely: it replaces the table rather than failing or duplicating it.

In [3]:
con.sql(f"CREATE OR REPLACE TABLE penguins AS SELECT * FROM read_csv('{DATA}')")
con.sql("SELECT count(*) AS row_count FROM penguins")

┌───────────┐
│ row_count │
│   int64   │
├───────────┤
│       344 │
└───────────┘

## 4. What did DuckDB decide about each column?

`DESCRIBE` shows each column's name and the **type** DuckDB guessed: `VARCHAR` is text, `BIGINT` is a whole number, `DOUBLE` is a decimal.
Look closely: does every guess make sense?

In [4]:
con.sql("DESCRIBE penguins")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ species           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_g       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ sex               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

### Something's wrong

`bill_length_mm` and `body_mass_g` are measurements, but DuckDB stored them as **text** (`VARCHAR`). Why?

Search the raw file for the letters `NA`: whoever recorded the data wrote `NA` where a value was missing. DuckDB saw letters in a number column and played it safe by treating the whole column as text. Text can't be averaged, so every calculation on these columns would fail.

This is the course in miniature: **the data arrived, but it wasn't ready.** The fix is to tell DuckDB that `NA` means "missing" (a `NULL` in database terms), then reload.

In [5]:
con.sql(f"""
    CREATE OR REPLACE TABLE penguins AS
    SELECT * FROM read_csv('{DATA}', nullstr = 'NA')
""")
con.sql("DESCRIBE penguins")

┌───────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name    │ column_type │  null   │   key   │ default │  extra  │
│      varchar      │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ species           │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ island            │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_length_mm    │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ bill_depth_mm     │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ flipper_length_mm │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ body_mass_g       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ sex               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ year              │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────────┴─────────────┴─────────┴─────────┴─────────┴─────────┘

Now the measurements are numbers (`DOUBLE` and `BIGINT`) and the missing values are real `NULL`s.

## 5. Ask questions with SQL

How many penguins of each species were measured, and how heavy are they on average?

In [27]:
con.sql("""
    SELECT island,
           count(*)                   AS penguins,
           round(avg(body_mass_g), 0) AS avg_mass_g,
           round(avg(bill_length_mm), 0) AS avg_bill_l,
           round(avg(bill_depth_mm), 0) AS avg_bill_d,
           round(avg(flipper_length_mm), 0) AS avg_flip_l
    FROM penguins
    GROUP BY island
    ORDER BY penguins DESC
""")

┌───────────┬──────────┬────────────┬────────────┬────────────┬────────────┐
│  island   │ penguins │ avg_mass_g │ avg_bill_l │ avg_bill_d │ avg_flip_l │
│  varchar  │  int64   │   double   │   double   │   double   │   double   │
├───────────┼──────────┼────────────┼────────────┼────────────┼────────────┤
│ Biscoe    │      168 │     4716.0 │       45.0 │       16.0 │      210.0 │
│ Dream     │      124 │     3713.0 │       44.0 │       18.0 │      193.0 │
│ Torgersen │       52 │     3706.0 │       39.0 │       18.0 │      191.0 │
└───────────┴──────────┴────────────┴────────────┴────────────┴────────────┘

Which values are missing? `count(column)` skips empty values, so subtracting it from `count(*)` gives the number missing.

In [28]:
con.sql("""
    SELECT count(*) - count(sex)          AS missing_sex,
           count(*) - count(body_mass_g)  AS missing_body_mass,
           count(*) - count(bill_length_mm) AS missing_bill_length,
           count(*) - count(flipper_length_mm) AS missing_flip_length
    FROM penguins
""")

┌─────────────┬───────────────────┬─────────────────────┬─────────────────────┐
│ missing_sex │ missing_body_mass │ missing_bill_length │ missing_flip_length │
│    int64    │       int64       │        int64        │        int64        │
├─────────────┼───────────────────┼─────────────────────┼─────────────────────┤
│          11 │                 2 │                   2 │                   2 │
└─────────────┴───────────────────┴─────────────────────┴─────────────────────┘

## 6. Your turn

Write **one query of your own** in the cell below. Some ideas:
- Which island has the most penguins? (`GROUP BY island`)
- Which species has the longest average flipper?
- How many penguins were measured each `year`?

Stuck? Copy a query from above and change one thing.

In [21]:
con.sql("""
    SELECT island,
           count(*)                   AS penguins,
    FROM penguins
    GROUP BY island
    ORDER BY penguins DESC
""")


┌───────────┬──────────┐
│  island   │ penguins │
│  varchar  │  int64   │
├───────────┼──────────┤
│ Biscoe    │      168 │
│ Dream     │      124 │
│ Torgersen │       52 │
└───────────┴──────────┘

## 7. Three observations

Double-click this cell to edit it. Write three things you noticed about the data. Good observations are specific and point to something you'd want to check before trusting the data.

1. 2 penguins have no recorded body mass
2. 2 penguins have no recorded bill length
3. 2 penguins have no recorded flipper length

*Example:* "11 penguins have no recorded sex. Before comparing males and females, I'd need to decide whether to drop them or keep them as 'unknown'." 

## 8. Save your work

1. Close the connection by running the cell below.
2. Commit and push: open **Source Control** in the left sidebar, type a message like `Lab 1 in class`, click **Commit**, then **Sync Changes**.

Your `lab1.duckdb` file is not uploaded. That's intentional: the database can always be rebuilt from this notebook.

**Next:** the at-home part is in `notebooks/lab1_home_your_data.ipynb`, where you do all of this again with a dataset you choose. See `labs/lab1.md` for the full Lab 1 checklist.

In [29]:
con.close()
print("Connection closed. Don't forget to commit and push.")

Connection closed. Don't forget to commit and push.
